In [1]:
import pandas as pd

# Interactions from NIH Percentile
Top 1% by RCR

In [2]:
df = pd.read_csv('../../../data/2026-07-07-test03-interaction_classifications.csv')
df.head()

,run_idx,prompt_version,temperature,used_raw_drug_candidates,seed_gene,pmid,block_id,skipped,skip_reason,candidate_drugs,...,raw_genes,raw_drug_count,raw_gene_count,error_message,drug,gene,interaction,interaction_type,directionality,evidence
0,0,v2,0,False,fix this,33495651,0,False,NaN,iron carbonyl;sucrose,...,NaN,3,0,No evidence of an interaction between iron car...,iron carbonyl,ALK,False,NaN,NaN,NaN
1,0,v2,0,False,fix this,33495651,0,False,NaN,iron carbonyl;sucrose,...,NaN,3,0,No evidence of interaction between sucrose and...,sucrose,ALK,False,NaN,NaN,NaN
2,0,v2,0,False,fix this,33597522,1,True,No drug and gene candidates,NaN,...,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,v2,0,False,fix this,33597522,2,True,No drug and gene candidates,NaN,...,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,v2,0,False,fix this,33277608,3,True,No drug and gene candidates,NaN,...,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
interactions = df[df['interaction']==True].reset_index(drop=True)
print(f'{len(interactions)} interactions identified in data set!')


1449 interactions identified in data set!


In [5]:
interactions[['drug','gene']].value_counts()

drug             gene
osimertinib      EGFR    44
sirolimus        MTOR    37
crizotinib       ALK     17
lorlatinib       ALK     17
dabrafenib       BRAF    17
                         ..
copper chloride  ATF3     1
                 DLAT     1
                 DLST     1
                 FDX1     1
zinc oxide       ULK1     1
Name: count, Length: 778, dtype: int64

In [6]:
import pandas as pd
import requests
from tqdm.auto import tqdm

GRAPHQL_URL = "https://dgidb.org/api/graphql"

QUERY = """
query($gene: String!) {
  genes(names: [$gene]) {
    nodes {
      name
      interactions {
        drug {
          name
          conceptId
        }
      }
    }
  }
}
"""

# ------------------------------------------------------------------
# Retrieve all drugs interacting with a gene in DGIdb
# ------------------------------------------------------------------

gene_cache = {}

for gene in tqdm(interactions["gene"].dropna().unique()):
    response = requests.post(
        GRAPHQL_URL,
        json={
            "query": QUERY,
            "variables": {"gene": gene}
        },
    )
    response.raise_for_status()

    data = response.json()

    drugs = set()

    try:
        nodes = data["data"]["genes"]["nodes"]
        if nodes:
            for interaction in nodes[0]["interactions"]:
                drug = interaction["drug"]
                if drug and drug["name"]:
                    drugs.add(drug["name"].strip().lower())
    except Exception:
        pass

    gene_cache[gene] = drugs

# ------------------------------------------------------------------
# Check every interaction
# ------------------------------------------------------------------

result = interactions.copy()

result["present_in_dgidb"] = result.apply(
    lambda row: row["drug"].strip().lower()
    in gene_cache.get(row["gene"], set()),
    axis=1,
)

result

  0%|          | 0/232 [00:00<?, ?it/s]

,run_idx,prompt_version,temperature,used_raw_drug_candidates,seed_gene,pmid,block_id,skipped,skip_reason,candidate_drugs,...,raw_drug_count,raw_gene_count,error_message,drug,gene,interaction,interaction_type,directionality,evidence,present_in_dgidb
0,0,v2,0,False,fix this,37597078,11,False,NaN,HYDROGEN PEROXIDE;NADPH;PEROXYNITRITE;glutathi...,...,12,13,NaN,HYDROGEN PEROXIDE,SOD1,True,enzyme substrate/target,inhibiting,NaN,False
1,0,v2,0,False,fix this,37597078,11,False,NaN,HYDROGEN PEROXIDE;NADPH;PEROXYNITRITE;glutathi...,...,12,13,NaN,PEROXYNITRITE,SOD1,True,substrate/target of antioxidant enzyme,inhibiting,NaN,False
2,0,v2,0,False,fix this,37597078,11,False,NaN,HYDROGEN PEROXIDE;NADPH;PEROXYNITRITE;glutathi...,...,12,13,NaN,hydrogen peroxide,SOD1,True,metabolism/degradation,inhibiting,NaN,False
3,0,v2,0,False,fix this,37597078,11,False,NaN,HYDROGEN PEROXIDE;NADPH;PEROXYNITRITE;glutathi...,...,12,13,NaN,oxygen,SOD1,True,enzyme activity modulation,activating,NaN,False
4,0,v2,0,False,fix this,33000412,63,False,NaN,anhydrous dextrose;cystine;glutamic acid;gluta...,...,5,2,NaN,anhydrous dextrose,SLC7A11,True,metabolic dependency/vulnerability,activating,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,0,v2,0,False,fix this,33357455,3551,False,NaN,cysteine;glutamic acid;glutathione,...,4,1,NaN,cysteine,GCLC,True,substrate,activating,NaN,False
1445,0,v2,0,False,fix this,33357455,3551,False,NaN,cysteine;glutamic acid;glutathione,...,4,1,NaN,glutamic acid,GCLC,True,substrate,activating,NaN,False
1446,0,v2,0,False,fix this,33357455,3551,False,NaN,cysteine;glutamic acid;glutathione,...,4,1,NaN,glutathione,GCLC,True,substrate,activating,NaN,False
1447,0,v2,0,False,fix this,35640743,3573,False,NaN,metformin;sirolimus,...,3,3,NaN,metformin,MTOR,True,inhibition,inhibitory,NaN,True


In [7]:
result["present_in_dgidb"].value_counts()

present_in_dgidb
False    726
True     723
Name: count, dtype: int64

In [11]:
novel = result.loc[~result["present_in_dgidb"]]
not_novel = result.loc[result["present_in_dgidb"]]
novel[["drug", "gene"]].drop_duplicates().sort_values(["gene", "drug"])
novel.drop_duplicates().sort_values(["gene", "drug"])

,run_idx,prompt_version,temperature,used_raw_drug_candidates,seed_gene,pmid,block_id,skipped,skip_reason,candidate_drugs,...,raw_drug_count,raw_gene_count,error_message,drug,gene,interaction,interaction_type,directionality,evidence,present_in_dgidb
459,0,v2,0,False,fix this,36203933,1458,False,NaN,cholesterol;dehydrated alcohol,...,4,11,NaN,dehydrated alcohol,ACACA,True,expression modulation,inhibiting,NaN,False
867,0,v2,0,False,fix this,33338751,2420,False,NaN,Broxuridine;flavanone;myricetin,...,6,18,NaN,myricetin,ACE,True,expression regulation,unclear,NaN,False
868,0,v2,0,False,fix this,33338751,2420,False,NaN,Broxuridine;flavanone;myricetin,...,6,18,NaN,myricetin,ACHE,True,expression regulation,unclear,NaN,False
505,0,v2,0,False,fix this,36675183,1659,False,NaN,Elesclomol;Tetrathiomolybdate;copper;copper ch...,...,10,11,NaN,Elesclomol,ACO2,True,downregulation,inhibitory,NaN,False
509,0,v2,0,False,fix this,36675183,1659,False,NaN,Elesclomol;Tetrathiomolybdate;copper;copper ch...,...,10,11,NaN,Tetrathiomolybdate,ACO2,True,copper chelation rescuing cuproptosis-associat...,activating,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,0,v2,0,False,fix this,37267363,2503,False,NaN,lactate;lysine;serine,...,4,11,NaN,lactate,UVRAG,True,indirect modulation via Vps34 lactylation,activating,NaN,False
1399,0,v2,0,False,fix this,33467546,3474,False,NaN,EMPA;anhydrous dextrose;cholesterol;empagliflozin,...,6,36,NaN,EMPA,XBP1,True,expression modulation,inhibitory,NaN,False
1413,0,v2,0,False,fix this,33467546,3474,False,NaN,EMPA;anhydrous dextrose;cholesterol;empagliflozin,...,6,36,NaN,empagliflozin,XBP1,True,expression modulation,inhibitory,NaN,False
481,0,v2,0,False,fix this,38975874,1559,False,NaN,MRTX1133;adagrasib;sotorasib,...,3,21,NaN,MRTX1133,YAP1,True,resistance association,activating,NaN,False


In [16]:
novel['evidence'].value_counts()

evidence
specific reductions in essential amino acids such as methionine...selectively impact...AMP-activated protein kinase (AMPK)                                                                                                                                                                                                                                                                                 1
Drugs like filgrastim, epigallocatechin gallate, curcumin, nicergoline and minocycline are under development which target these pathways and modulate the disease condition... It activates various proteins including Bcl-2 family proteins like Bax, Bad, Bid...                                                                                                                                         1
The apoptotic elements interact with trophic factors, signaling molecules including...BDNF/TrkB/CREB...Drugs like...curcumin...are under development which target these pathways and modulate the dis

In [17]:
# pmid drug gene interaction interaction_type directionality 
novel[['pmid','drug','gene','interaction','interaction_type','directionality','evidence']].to_csv('../../../data/2026-07-08-test03-novel-interactions.csv')
novel[['pmid','drug','gene','interaction','interaction_type','directionality','evidence']]

,pmid,drug,gene,interaction,interaction_type,directionality,evidence
0,37597078,HYDROGEN PEROXIDE,SOD1,True,enzyme substrate/target,inhibiting,NaN
1,37597078,PEROXYNITRITE,SOD1,True,substrate/target of antioxidant enzyme,inhibiting,NaN
2,37597078,hydrogen peroxide,SOD1,True,metabolism/degradation,inhibiting,NaN
3,37597078,oxygen,SOD1,True,enzyme activity modulation,activating,NaN
4,33000412,anhydrous dextrose,SLC7A11,True,metabolic dependency/vulnerability,activating,NaN
...,...,...,...,...,...,...,...
1442,33357455,glutamic acid,GCLC,True,substrate,activating,NaN
1443,33357455,glutathione,GCLC,True,substrate relationship / biosynthesis,activating,NaN
1444,33357455,cysteine,GCLC,True,substrate,activating,NaN
1445,33357455,glutamic acid,GCLC,True,substrate,activating,NaN


In [18]:
not_novel[['pmid','drug','gene','interaction','interaction_type','directionality','evidence']].to_csv('../../../data/2026-07-08-test03-known-interactions.csv')
not_novel[['pmid','drug','gene','interaction','interaction_type','directionality','evidence']]

,pmid,drug,gene,interaction,interaction_type,directionality,evidence
9,38319329,cabozantinib s-malate,RET,True,inhibitor,inhibitory,NaN
10,38319329,dabrafenib,BRAF,True,targeted therapy,inhibiting,NaN
11,38319329,dabrafenib,BRAF,True,targeted therapy,inhibiting,NaN
12,38319329,selpercatinib,RET,True,targeted therapy,inhibiting,NaN
15,37779156,sirolimus,MTOR,True,inhibitor,inhibitory,NaN
...,...,...,...,...,...,...,...
1431,37161388,gefitinib,EGFR,True,resistance modulation,ambiguous,NaN
1432,37161388,gefitinib,EGFR,True,resistance,inhibitory,NaN
1435,37161388,gefitinib,EGFR,True,resistance,inhibitory,NaN
1447,35640743,metformin,MTOR,True,inhibition,inhibitory,NaN
